In [21]:
import functional_functions as ff
from nilearn.maskers import SurfaceLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn.surface import SurfaceImage
import numpy as np
import pandas as pd
from pathlib import Path

In [22]:
# create mesh and parcellate based on MMP
labels_image = ff.create_mmp_mesh()

In [23]:
# create look up table with labels 
lut = ff.create_mmp_lookup()

In [24]:
# make and fit the masker to the ROIs from Glasser 
surface_lbls_masker = SurfaceLabelsMasker(labels_img=labels_image, lut=lut).fit()

In [151]:
# read in participant csv 
subjects = ['197348', '744553', '530635']
subjects

['197348', '744553', '530635']

In [152]:
# instantiate storage matrix 
mat = []

In [153]:
# loop through all participants 
for sub_num in subjects:
    
    (all_left_hemi, all_right_hemi) = ff.ciis_to_giis(sub_num)
    
    # put our data in SurfaceImage object 
    func_data = SurfaceImage(
        mesh=labels_image.mesh,
        data= {
            "left": all_left_hemi.T,
            "right": all_right_hemi.T
        }
    )
    roi_time_series = surface_lbls_masker.transform(func_data)
    
    # make correlation matrix 
    correlation_measure = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)
    correlation_matrix = correlation_measure.fit_transform([roi_time_series])[0]

    # add subject number to beginning of vectorized corrmat
    # add to matrix
    mat.append(np.insert(correlation_matrix, 0, sub_num))
    print(sub_num)
    

pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_1549/3392555806.py:18: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix = correlation_measure.fit_transform([roi_time_series])[0]


197348


pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_1549/3392555806.py:18: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix = correlation_measure.fit_transform([roi_time_series])[0]


744553


pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1


530635


/tmp/ipykernel_1549/3392555806.py:18: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix = correlation_measure.fit_transform([roi_time_series])[0]


In [156]:
np.shape(mat)

(3, 64621)

In [157]:
mat_without_id = []

for sub in mat:
    mat_without_id.append(sub[1:])

np.shape(mat_without_id)

(3, 64620)

# Prepare the data

## z-transform the matrices

In [158]:
mat_stack = np.stack(mat_without_id)
mat_stack

array([[ 0.86223075, -0.38793825, -0.47694481, ..., -0.91643542,
        -0.1138736 ,  0.95911863],
       [ 0.69776761,  0.88059598,  0.581369  , ..., -0.92561786,
         0.86232853,  0.94395767],
       [ 0.55152037,  0.69650391,  0.65150539, ...,  0.89554579,
         0.74630144,  0.96072585]])

In [159]:
mat_zScore = np.arctanh(mat_stack)
np.shape(mat_zScore)

(3, 64620)

## Load the sociodemographics

In [288]:
import pandas as pd

test_df = pd.read_csv("test_split.csv")
dd_df = pd.read_csv("DD_AUC_targets.csv")

In [289]:
subjects = [int(sub) for sub in subjects]

test_df_DD = test_df.merge(
    dd_df[["Subject", "DDisc_AUC_40K"]],
    on="Subject",
    how="left"
)

test_subjects = test_df_DD[test_df_DD["Subject"].isin(subjects)]
test_subjects = test_subjects[["Subject", "Gender", "Age", "DDisc_AUC_40K"]]
test_subjects

,Subject,Gender,Age,DDisc_AUC_40K
0,197348,F,26-30,0.595703
1,744553,F,31-35,0.508073
2,530635,M,26-30,0.931250


## Merge corr matraces with sociodemographics

In [290]:
df_mat_zScores = pd.DataFrame(mat_zScore)
df_mat_zScores

,0,1,2,3,4,5,6,7,8,9,...,64610,64611,64612,64613,64614,64615,64616,64617,64618,64619
0,1.301975,-0.409371,-0.519022,1.818193,1.464141,-0.553572,1.660646,1.521789,-0.573703,2.172369,...,1.984265,1.961369,2.011340,-1.307030,-1.532273,1.465225,1.063169,-1.566301,-0.114370,1.934788
1,0.862937,1.378416,0.664528,0.939572,0.923052,0.710383,0.434622,0.761565,0.323616,1.149830,...,0.926417,1.630380,1.780142,-1.326744,-1.192937,-1.071310,1.203622,-1.626893,1.302356,1.773187
2,0.620564,0.860478,0.777910,1.073321,0.366872,0.470017,0.943462,0.876576,2.017596,0.512867,...,2.023508,1.929381,-0.403257,-1.541660,-0.970913,1.629269,1.895527,1.449257,0.964554,1.955252


In [291]:
merged_df = pd.concat([test_subjects, df_mat_zScores], axis=1)
merged_df

,Subject,Gender,Age,DDisc_AUC_40K,0,1,2,3,4,5,...,64610,64611,64612,64613,64614,64615,64616,64617,64618,64619
0,197348,F,26-30,0.595703,1.301975,-0.409371,-0.519022,1.818193,1.464141,-0.553572,...,1.984265,1.961369,2.011340,-1.307030,-1.532273,1.465225,1.063169,-1.566301,-0.114370,1.934788
1,744553,F,31-35,0.508073,0.862937,1.378416,0.664528,0.939572,0.923052,0.710383,...,0.926417,1.630380,1.780142,-1.326744,-1.192937,-1.071310,1.203622,-1.626893,1.302356,1.773187
2,530635,M,26-30,0.931250,0.620564,0.860478,0.777910,1.073321,0.366872,0.470017,...,2.023508,1.929381,-0.403257,-1.541660,-0.970913,1.629269,1.895527,1.449257,0.964554,1.955252


# Analysis

## Define the predictors

In [293]:
demographic_vars = ['Gender', 'Age']
connectivity_vars = list(range(64620))

## Transform ordinal data

In [294]:
age_ranges = df["Age"].unique()
age_ranges

array(['26-30', '22-25', '31-35', '36+'], dtype=object)

In [295]:
age_mapping = {age: i for i, age in enumerate(age_ranges)}
merged_df["Age_ordinal"] = merged_df["Age"].map(age_mapping)
merged_df

,Subject,Age,DDisc_AUC_40K,0,1,2,3,4,5,6,...,64612,64613,64614,64615,64616,64617,64618,64619,Gender_M,Age_ordinal
0,197348,26-30,0.595703,1.301975,-0.409371,-0.519022,1.818193,1.464141,-0.553572,1.660646,...,2.011340,-1.307030,-1.532273,1.465225,1.063169,-1.566301,-0.114370,1.934788,False,0
1,744553,31-35,0.508073,0.862937,1.378416,0.664528,0.939572,0.923052,0.710383,0.434622,...,1.780142,-1.326744,-1.192937,-1.071310,1.203622,-1.626893,1.302356,1.773187,False,2
2,530635,26-30,0.931250,0.620564,0.860478,0.777910,1.073321,0.366872,0.470017,0.943462,...,-0.403257,-1.541660,-0.970913,1.629269,1.895527,1.449257,0.964554,1.955252,True,0


## Create dummy variables for categorical data

In [296]:
# Define categorical variables
categorical_cols = ['Gender']

# Dummy code
merged_df = pd.get_dummies(
    merged_df,
    columns=categorical_cols,
    drop_first=True
)

merged_df

KeyError: "None of [Index(['Gender'], dtype='object')] are in the [columns]"

## Elastic Net Pipeline

### Load sklearn modules

In [297]:
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

In [298]:
feature_cols = ['Age_ordinal', 'Gender_M'] + connectivity_vars

In [299]:
X = merged_df[feature_cols]
y = merged_df["DDisc_AUC_40K"]

## K-Fold Parameters

In [301]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# store out-of-fold predictions here — one prediction per subject
oof_predictions = np.full(len(X), np.nan)
fold_r2_scores = []

## Quickcheck the shape of the data

In [302]:
print(X.shape, y.shape)
print(X.index.equals(y.index))

(3, 64622) (3,)
True


# Run over folds

In [304]:
fold_coefs = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_tr[continuous_cols] = scaler.fit_transform(X_tr[continuous_cols])
    X_val[continuous_cols] = scaler.transform(X_val[continuous_cols])

    # Tuning happens HERE, using only this fold's training data
    model = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                          alphas=np.logspace(-3, 1, 50),
                          cv=5, random_state=42, max_iter=10000)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_val)
    oof_predictions[val_idx] = preds
    fold_r2_scores.append(model.score(X_val, y_val))
    print(f"Fold {fold_num + 1}: R² = {fold_r2_scores[-1]:.4f}, "
          f"alpha={model.alpha_:.4f}, l1_ratio={model.l1_ratio_}")
    fold_coefs.append(model.coef_)

print(f"Mean CV R²: {np.mean(fold_r2_scores):.4f} (+/- {np.std(fold_r2_scores):.4f})")

ValueError: Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=3.

## Check overall out-of-fold correlation

In [306]:
from scipy.stats import pearsonr
r, p_val = pearsonr(y, oof_predictions)
print(f"Out-of-fold r = {r:.3f}, p = {p_val:.3f}")

Out-of-fold r = nan, p = nan


# Extract coefficient from the final model


In [308]:
fold_coef_df = pd.DataFrame(fold_coefs, columns=feature_cols)
print(fold_coef_df.describe())  # mean and std of each coefficient across folds

       Age_ordinal Gender_M    0    1    2    3    4    5    6    7  ...  \
count            0        0    0    0    0    0    0    0    0    0  ...   
unique           0        0    0    0    0    0    0    0    0    0  ...   
top            NaN      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...   
freq           NaN      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...   

       64610 64611 64612 64613 64614 64615 64616 64617 64618 64619  
count      0     0     0     0     0     0     0     0     0     0  
unique     0     0     0     0     0     0     0     0     0     0  
top      NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
freq     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  

[4 rows x 64622 columns]


## Plot the coefficients

In [ ]:
# Build a summary table: mean and std of each feature's coefficient across folds
coef_summary = pd.DataFrame({
    'feature': feature_cols,
    'mean_coef': fold_coef_df.mean(),
    'std_coef': fold_coef_df.std()
}).sort_values('mean_coef', ascending=False)

# Plot
plt.figure(figsize=(8, 6))
plt.barh(coef_summary['feature'], coef_summary['mean_coef'],
         xerr=coef_summary['std_coef'], color='steelblue', capsize=3)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (mean ± SD across folds)')
plt.title('Elastic Net Coefficients Across Folds')
plt.gca().invert_yaxis()  # highest at top
plt.tight_layout()
plt.show()